In [10]:
import pandas as pd
import numpy as np
from prefixspan import PrefixSpan
import importlib
import sys

# Reload modules to get latest changes
if 'algo.base_algo' in sys.modules:
    importlib.reload(sys.modules['algo.base_algo'])
if 'algo.gsp' in sys.modules:
    importlib.reload(sys.modules['algo.gsp'])

from algo.gsp import GSPAlgo
from data.base_data import Data

In [11]:
# Load raw cancer dataset using Data class
data_obj = Data("cancer")
original_df = data_obj.get_data()
print(f"Loaded raw dataset: {original_df.shape}")
original_df.head()

Loaded raw dataset: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [12]:
def convert_dataframe_to_sequences(df, top_k=3):
    """
    Convert DataFrame from GSPAlgo (with feature_1_name, feature_1_value, etc.) 
    to sequences format for PrefixSpan.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: patient_id, diagnosis, feature_1_name, feature_1_value, ...
    top_k : int
        Number of top-K features to include (K for feature transformation)
    Returns:
    --------
    tuple: (malignant_sequences, benign_sequences)
        Lists of sequences for malignant and benign patients
    """
    malignant_sequences = []
    benign_sequences = []
    
    for idx, row in df.iterrows():
        # Create sequence from feature_1, feature_2, ..., feature_K
        sequence = []
        for i in range(1, top_k + 1):
            name_col = f'feature_{i}_name'
            value_col = f'feature_{i}_value'
            
            if name_col in row.index and value_col in row.index:
                name = row[name_col]
                value = row[value_col]
                
                # Skip empty features
                if pd.notna(name) and pd.notna(value) and name != '' and value != '':
                    item = f"{value}_{name}"
                    sequence.append((item,))
        
        # Add to appropriate list
        if row['diagnosis'] == 'M':
            malignant_sequences.append(sequence)
        elif row['diagnosis'] == 'B':
            benign_sequences.append(sequence)
    
    return malignant_sequences, benign_sequences

In [13]:
def mine_patterns(sequences, maxlen=2, min_sup=0.02):
    """
    Mine frequent sequential patterns using PrefixSpan.
    
    Parameters:
    -----------
    sequences : list
        List of sequences (each sequence is a list of tuple itemsets)
    maxlen : int
        Maximum length of patterns to mine
    min_sup : float
        Minimum support as a fraction (e.g., 0.02 = 2%)
    
    Returns:
    --------
    list: List of (support, pattern) tuples
    """
    if not sequences:
        return []
    
    support_count = max(1, int(min_sup * len(sequences)))
    
    try:
        ps = PrefixSpan(sequences)
        ps.minlen = 1  # Allow patterns of length 1
        frequent_patterns = ps.frequent(support_count)
        
        # Filter patterns by maxlen (pattern length = number of itemsets)
        filtered_patterns = [
            (support, pattern) for support, pattern in frequent_patterns
            if len(pattern) <= maxlen
        ]
        
        return filtered_patterns
    except Exception as e:
        print(f"Error mining patterns: {e}")
        return []


In [14]:
# Initialize GSPAlgo (this will handle discretization internally)
gsp = GSPAlgo(data_obj)


In [15]:
# Define parameter grid
strategies = ['uniform', 'quantile', 'kmeans']
K_values = [5, 10, 15]
L_values = [4, 6, 8]
min_sup_values = [0.20, 0.15, 0.10]

# Store results
results = []

print("Starting parameter grid search...")
print(f"Total combinations: {len(strategies)} strategies × {len(K_values)} K values × {len(L_values)} L values × {len(min_sup_values)} min_sup values")
print(f"= {len(strategies) * len(K_values) * len(L_values) * len(min_sup_values)} total runs\n")

for strategy in strategies:
    print(f"Processing strategy: {strategy}")
    
    # Step 1 & 2: Generate sequences per patient for each K value
    # GSPAlgo._generate_sequences_for_strategy handles discretization and z-score ranking
    for K in K_values:
        print(f"  K={K}: Generating sequences...", end=" ")
        # Generate sequences with the specific K value (top_k=K for feature transformation)
        sequences_df =  gsp._generate_sequences_for_strategy(strategy, top_k=K)
        sequences_mal, sequences_ben = convert_dataframe_to_sequences(sequences_df, top_k=K)
        print(f"Malignant: {len(sequences_mal)}, Benign: {len(sequences_ben)}")
        
        # Step 3: Mine patterns using PrefixSpan
        for min_sup in min_sup_values:
            for L in L_values:
                # Mine patterns for malignant and benign
                # maxlen=L is for pattern mining (maximum pattern length)
                patterns_m = mine_patterns(sequences_mal, maxlen=L, min_sup=min_sup)
                patterns_b = mine_patterns(sequences_ben, maxlen=L, min_sup=min_sup)
                
                # Record results
                results.append({
                    'strategy': strategy,
                    'top-K features K': K,
                    'min_sup': min_sup,
                    'max sequence length L': L,
                    'malignant_patterns': len(patterns_m),
                    'benign_patterns': len(patterns_b),
                    'total_patterns': len(patterns_m) + len(patterns_b)
                })

print(f"\nCompleted {len(results)} runs!")


Starting parameter grid search...
Total combinations: 3 strategies × 3 K values × 3 L values × 3 min_sup values
= 81 total runs

Processing strategy: uniform
  K=5: Generating sequences... Malignant: 212, Benign: 357
  K=10: Generating sequences... Malignant: 212, Benign: 357
  K=15: Generating sequences... Malignant: 212, Benign: 357
Processing strategy: quantile
  K=5: Generating sequences... Malignant: 212, Benign: 357
  K=10: Generating sequences... Malignant: 212, Benign: 357
  K=15: Generating sequences... Malignant: 212, Benign: 357
Processing strategy: kmeans
  K=5: Generating sequences... Malignant: 212, Benign: 357
  K=10: Generating sequences... Malignant: 212, Benign: 357
  K=15: Generating sequences... Malignant: 212, Benign: 357

Completed 81 runs!


In [16]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df


,strategy,top-K features K,min_sup,max sequence length L,malignant_patterns,benign_patterns,total_patterns
0,uniform,5,0.20,4,1,7,8
1,uniform,5,0.20,6,1,7,8
2,uniform,5,0.20,8,1,7,8
3,uniform,5,0.15,4,8,13,21
4,uniform,5,0.15,6,8,13,21
...,...,...,...,...,...,...,...
76,kmeans,15,0.15,6,101,152,253
77,kmeans,15,0.15,8,101,152,253
78,kmeans,15,0.10,4,230,418,648
79,kmeans,15,0.10,6,230,419,649
